# Public portfolio adaptation
Adapted from my Fall 2025 Gilead project. Outputs and original data are excluded.
Company examples are fictional. Requires your own Databricks workspace and credentials.
This notebook is retained as research code; see the repository documentation for limitations.


# Few-Shot Testing: Claude and Llama

This notebook tests different few-shot prompting strategies across two models:
- **Claude**: Anthropic's Claude model via API
- **Llama**: Meta's Llama-3.1-8b-instruct model via Databricks endpoint

**Window Size**: 1 chunk [n] - Current sentence only

**Data Limit**: Testing on first 100 sentences only (for faster iteration)

**Experiments**:
- Zero-shot: No examples
- One-shot: 1 positive example
- Two-shot: 1 positive + 1 negative example
- Three-shot: 1 positive + 1 negative + 1 neutral example

**Score Values**:
- **NA (null)**: Only at boundaries where context is missing
- **0.0**: Only when model explicitly returns 0.0 for objective (non-commentary) facts
- **"Error"**: When model does not produce a valid response (timeout, parsing errors, invalid scores, missing responses, etc.)



In [ ]:
# Install required packages
%pip install anthropic transformers torch accelerate mlflow python-dotenv
%restart_python



In [ ]:
# ==================================================
# IMPORTS AND SETUP
# ==================================================
import anthropic
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
import json
import re
# import torch  # Commented out - not needed for Llama
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline  # Commented out - not needed
from mlflow.deployments import get_deploy_client
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
import os
from dotenv import load_dotenv

spark = SparkSession.builder.getOrCreate()

print("✅ All imports successful")



In [ ]:
# ==================================================
# MODEL SETUP
# ==================================================

# Claude setup
# Load environment variables from .env file
# Create a .env file in the Databricks folder with: CLAUDE_API_KEY=your-api-key-here
load_dotenv()

api_key = os.getenv("CLAUDE_API_KEY")
if not api_key:
    raise ValueError("CLAUDE_API_KEY not found in environment. Please create a .env file with CLAUDE_API_KEY=your-api-key-here")

claude_client = anthropic.Anthropic(api_key=api_key)

# Databricks endpoints
db_client = get_deploy_client("databricks")

# Llama endpoint
LLAMA = "databricks-meta-llama-3-1-8b-instruct"

print(f"\n✅ Model configurations set")
print(f"   Claude: Anthropic API")
print(f"   Llama endpoint: {LLAMA}")



In [ ]:
# ==================================================
# HELPER FUNCTIONS
# ==================================================
def polarity_to_1_7(p_pos: float, p_neg: float) -> float:
    """Convert polarity scores to 1-7 scale"""
    return float(max(1.0, min(7.0, 4.0 + 3.0 * (p_pos - p_neg))))

def extract_json(text):
    """Robust JSON extraction from model responses - matches Combined_Sentiment_Analysis exactly
    ALWAYS returns a dict with 'score' key - never returns None"""
    # Handle None or non-string input
    if text is None or not isinstance(text, str):
        return {"score": 4}
    
    # Try to parse as complete JSON
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict) and "score" in parsed:
            return parsed
    except:
        pass
    
    # Try to find JSON object with score
    match = re.search(r'\{[^}]*"score"[^}]*\}', text)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, dict):
                return parsed
        except:
            pass
    
    # Try to extract just the score number
    number_match = re.search(r'"score":\s*([0-9.]+)', text)
    if number_match:
        try:
            score_val = float(number_match.group(1))
            return {"score": score_val}
        except:
            pass
    
    # ALWAYS return a dict - never None (matches working version)
    return {"score": 4}

print("✅ Helper functions defined")



In [ ]:
# ==================================================
# LOAD & CLEAN DATA (LIMITED TO FIRST 100 ROWS)
# ==================================================
print("Loading data from parsed_data table...")
df = spark.table("parsed_data")

src = (df
    .withColumn("row_idx", F.monotonically_increasing_id())
    .withColumn("report_id", F.col("`doc #`").cast("long"))
    .withColumn("sentence_number", F.col("`sentence #`").cast("long"))
    .withColumn("sentence", F.substring(F.col("context"), 1, 2000))
    .filter(F.col("report_id").isNotNull())
    .select("report_id", "row_idx", "sentence_number", "sentence")
    .orderBy("report_id", "row_idx")
    .limit(100)  # Limit to first 100 rows for faster testing
)

print(f"✅ Data loaded: {src.count()} sentences (limited to first 100 for testing)")



In [ ]:
# ==================================================
# CREATE WINDOW SIZE 1 [n]
# ==================================================
print("Creating window size 1 [n]...")

# Create window 1: [n] - just the current sentence
windows_1chunk = src.select(
    "report_id", "sentence_number",
    F.col("sentence").alias("window_text")
)

print(f"✅ Window 1 created: {windows_1chunk.count()} windows")
display(windows_1chunk.limit(5))

# Convert to list for processing (limited to first 100 rows)
windows_list = windows_1chunk.collect()
print(f"✅ Converted to list: {len(windows_list)} windows ready for processing")



In [ ]:
# ==================================================
# DEFINE FEW-SHOT EXAMPLES
# ==================================================
EXAMPLE_POSITIVE = {
    "text": "Example Company raised its annual outlook following stronger demand.",
    "score": 6.50,
    "label": "bullish/optimistic commentary"
}

EXAMPLE_NEGATIVE = {
    "text": "Example Company lowered its outlook as competition increased.",
    "score": 2.25,
    "label": "pessimistic/bearish commentary"
}

EXAMPLE_NEUTRAL = {
    "text": "Example Company scheduled an investor update for next month.",
    "score": 4.00,
    "label": "neutral commentary"
}

print("✅ Examples defined:")
print(f"  Positive: {EXAMPLE_POSITIVE['text'][:80]}... → {EXAMPLE_POSITIVE['score']}")
print(f"  Negative: {EXAMPLE_NEGATIVE['text'][:80]}... → {EXAMPLE_NEGATIVE['score']}")
print(f"  Neutral: {EXAMPLE_NEUTRAL['text'][:80]}... → {EXAMPLE_NEUTRAL['score']}")



In [ ]:
# ==================================================
# DEFINE PROMPTS FOR EACH FEW-SHOT LEVEL
# ==================================================

def create_prompt_zero_shot():
    return (
        "You are rating Wall Street financial analysis on Example Company stock performance (EXAMPLE). "
        "Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. "
        "You might get mentions of Example Company's competitors' performance, but remember to rate on EXAMPLE performance only.\n\n"
        "Return ONLY a valid JSON array, where each item is {{\"id\":\"<id>\",\"score\":<decimal>}}.\n"
        "Scores must be decimal numbers with 3 significant figures (e.g., 2.35, 3.67, 4.51, 5.83, 6.42), ranging from 0.0 to 7.0.\n"
        "Scale: 0.0 = objective (NON-commentary), 1.0 = very pessimistic, 4.0 = neutral commentary, 7.0 = very optimistic.\n"
        "Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary.\n"
        "Use the FULL range with granular decimal precision - do NOT default to whole numbers. Use distinct scores (e.g., 4.23, 4.51, 4.67) rather than repeating the same values.\n\n"
        "Items:\n{items}\n"
        "Respond with ONLY the JSON array."
    )

def create_prompt_one_shot():
    return (
        "You are rating Wall Street financial analysis on Example Company stock performance (EXAMPLE). "
        "Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. "
        "You might get mentions of Example Company's competitors' performance, but remember to rate on EXAMPLE performance only.\n\n"
        "Return ONLY a valid JSON array, where each item is {{\"id\":\"<id>\",\"score\":<decimal>}}.\n"
        "Scores must be decimal numbers with 3 significant figures (e.g., 2.35, 3.67, 4.51, 5.83, 6.42), ranging from 0.0 to 7.0.\n"
        "Scale: 0.0 = objective (NON-commentary), 1.0 = very pessimistic, 4.0 = neutral commentary, 7.0 = very optimistic.\n"
        "Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary.\n"
        "Use the FULL range with granular decimal precision - do NOT default to whole numbers. Use distinct scores (e.g., 4.23, 4.51, 4.67) rather than repeating the same values.\n\n"
        f"Here is an example of {EXAMPLE_POSITIVE['label']} on EXAMPLE:\n"
        f"Example text: {EXAMPLE_POSITIVE['text']}\n"
        f"Example score: {EXAMPLE_POSITIVE['score']:.2f} ({EXAMPLE_POSITIVE['label']} with granular decimal precision)\n\n"
        "Items:\n{items}\n"
        "Respond with ONLY the JSON array."
    )

def create_prompt_two_shot():
    return (
        "You are rating Wall Street financial analysis on Example Company stock performance (EXAMPLE). "
        "Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. "
        "You might get mentions of Example Company's competitors' performance, but remember to rate on EXAMPLE performance only.\n\n"
        "Return ONLY a valid JSON array, where each item is {{\"id\":\"<id>\",\"score\":<decimal>}}.\n"
        "Scores must be decimal numbers with 3 significant figures (e.g., 2.35, 3.67, 4.51, 5.83, 6.42), ranging from 0.0 to 7.0.\n"
        "Scale: 0.0 = objective (NON-commentary), 1.0 = very pessimistic, 4.0 = neutral commentary, 7.0 = very optimistic.\n"
        "Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary.\n"
        "Use the FULL range with granular decimal precision - do NOT default to whole numbers. Use distinct scores (e.g., 4.23, 4.51, 4.67) rather than repeating the same values.\n\n"
        "Here are examples:\n"
        f"Example 1 ({EXAMPLE_POSITIVE['label']}): {EXAMPLE_POSITIVE['text']} → Score: {EXAMPLE_POSITIVE['score']:.2f}\n"
        f"Example 2 ({EXAMPLE_NEGATIVE['label']}): {EXAMPLE_NEGATIVE['text']} → Score: {EXAMPLE_NEGATIVE['score']:.2f}\n\n"
        "Items:\n{items}\n"
        "Respond with ONLY the JSON array."
    )

def create_prompt_three_shot():
    return (
        "You are rating Wall Street financial analysis on Example Company stock performance (EXAMPLE). "
        "Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. "
        "You might get mentions of Example Company's competitors' performance, but remember to rate on EXAMPLE performance only.\n\n"
        "Return ONLY a valid JSON array, where each item is {{\"id\":\"<id>\",\"score\":<decimal>}}.\n"
        "Scores must be decimal numbers with 3 significant figures (e.g., 2.35, 3.67, 4.51, 5.83, 6.42), ranging from 0.0 to 7.0.\n"
        "Scale: 0.0 = objective (NON-commentary), 1.0 = very pessimistic, 4.0 = neutral commentary, 7.0 = very optimistic.\n"
        "Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary.\n"
        "Use the FULL range with granular decimal precision - do NOT default to whole numbers. Use distinct scores (e.g., 4.23, 4.51, 4.67) rather than repeating the same values.\n\n"
        "Here are examples:\n"
        f"Example 1 ({EXAMPLE_POSITIVE['label']}): {EXAMPLE_POSITIVE['text']} → Score: {EXAMPLE_POSITIVE['score']:.2f}\n"
        f"Example 2 ({EXAMPLE_NEGATIVE['label']}): {EXAMPLE_NEGATIVE['text']} → Score: {EXAMPLE_NEGATIVE['score']:.2f}\n"
        f"Example 3 ({EXAMPLE_NEUTRAL['label']}): {EXAMPLE_NEUTRAL['text']} → Score: {EXAMPLE_NEUTRAL['score']:.2f}\n\n"
        "Items:\n{items}\n"
        "Respond with ONLY the JSON array."
    )

print("✅ Prompts defined for all few-shot levels")



In [ ]:
# ==================================================
# LLAMA SENTIMENT ANALYSIS - FEW-SHOT PROMPTS
# ==================================================
print("🦙 Running Llama sentiment analysis with few-shot prompts...")

def create_llama_prompt_zero_shot():
    return (
        'Financial sentiment analysis on Example Company stock performance (EXAMPLE). '
        'Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. '
        'Return ONLY strict JSON: {\\"id\\":\\"<id>\\",\\"score\\":<decimal>}. '
        'Scores must be decimal numbers ranging from 0.0 to 7.0. '
        'Scale: 0.0=objective (NON-commentary), 1.0=very pessimistic, 4.0=neutral commentary, 7.0=very optimistic. '
        'Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary. '
        'Text: '
    )

def create_llama_prompt_one_shot():
    return (
        'Financial sentiment analysis on Example Company stock performance (EXAMPLE). '
        'Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. '
        f'Example ({EXAMPLE_POSITIVE["label"]}): {EXAMPLE_POSITIVE["text"]} → Score: {EXAMPLE_POSITIVE["score"]:.2f}. '
        'Return ONLY strict JSON: {\\"id\\":\\"<id>\\",\\"score\\":<decimal>}. '
        'Scores must be decimal numbers ranging from 0.0 to 7.0. '
        'Scale: 0.0=objective (NON-commentary), 1.0=very pessimistic, 4.0=neutral commentary, 7.0=very optimistic. '
        'Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary. '
        'Text: '
    )

def create_llama_prompt_two_shot():
    return (
        'Financial sentiment analysis on Example Company stock performance (EXAMPLE). '
        'Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. '
        f'Example 1 ({EXAMPLE_POSITIVE["label"]}): {EXAMPLE_POSITIVE["text"]} → Score: {EXAMPLE_POSITIVE["score"]:.2f}. '
        f'Example 2 ({EXAMPLE_NEGATIVE["label"]}): {EXAMPLE_NEGATIVE["text"]} → Score: {EXAMPLE_NEGATIVE["score"]:.2f}. '
        'Return ONLY strict JSON: {\\"id\\":\\"<id>\\",\\"score\\":<decimal>}. '
        'Scores must be decimal numbers ranging from 0.0 to 7.0. '
        'Scale: 0.0=objective (NON-commentary), 1.0=very pessimistic, 4.0=neutral commentary, 7.0=very optimistic. '
        'Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary. '
        'Text: '
    )

def create_llama_prompt_three_shot():
    return (
        'Financial sentiment analysis on Example Company stock performance (EXAMPLE). '
        'Rate whether the given information is bullish and optimistic, neutral, or bearish and pessimistic on EXAMPLE. '
        f'Example 1 ({EXAMPLE_POSITIVE["label"]}): {EXAMPLE_POSITIVE["text"]} → Score: {EXAMPLE_POSITIVE["score"]:.2f}. '
        f'Example 2 ({EXAMPLE_NEGATIVE["label"]}): {EXAMPLE_NEGATIVE["text"]} → Score: {EXAMPLE_NEGATIVE["score"]:.2f}. '
        f'Example 3 ({EXAMPLE_NEUTRAL["label"]}): {EXAMPLE_NEUTRAL["text"]} → Score: {EXAMPLE_NEUTRAL["score"]:.2f}. '
        'Return ONLY strict JSON: {\\"id\\":\\"<id>\\",\\"score\\":<decimal>}. '
        'Scores must be decimal numbers ranging from 0.0 to 7.0. '
        'Scale: 0.0=objective (NON-commentary), 1.0=very pessimistic, 4.0=neutral commentary, 7.0=very optimistic. '
        'Use 0.0 ONLY for objective facts; use 1.0–7.0 for subjective/opinionated commentary. '
        'Text: '
    )

def score_llama_with_retry(row, prompt_template, max_retries=3):
    """Score with retry logic and exponential backoff for Llama"""
    if row.window_text is None:
        return {
            "report_id": row.report_id,
            "sentence_number": row.sentence_number,
            "llama_score": None
        }
    
    for attempt in range(max_retries):
        try:
            full_prompt = prompt_template + f"[{row.report_id}|{row.sentence_number}]: {row.window_text[:2000]}"
            # Reduce max_tokens and add timeout handling
            response = db_client.predict(
                endpoint=LLAMA,
                inputs={
                    "messages": [{"role": "user", "content": full_prompt}],
                    "temperature": 0.0,
                    "max_tokens": 30  # Reduced from 50 to speed up responses
                }
            )
            
            # Extract content - matches working version exactly
            raw_content = response["choices"][0]["message"]["content"]
            
            # extract_json always returns a dict (never None) - defaults to {"score": 4} if parsing fails
            # Wrap in try/except as final safety net
            try:
                result = extract_json(raw_content)
            except Exception:
                result = {"score": 4}
            
            # Defensive check - ensure result is always a dict (should never be None, but just in case)
            if result is None or not isinstance(result, dict):
                result = {"score": 4}
            
            # Safely get score with default - multiple layers of protection
            try:
                score = float(result.get("score", 4))
            except (AttributeError, TypeError, ValueError):
                score = 4.0
            
            return {
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "llama_score": score
            }
            
        except Exception as e:
            error_str = str(e)
            # If timeout, return default score immediately instead of retrying
            if "timeout" in error_str.lower() or "timed out" in error_str.lower():
                print(f"Llama timeout for [{row.report_id}|{row.sentence_number}]: returning default score 4")
                return {
                    "report_id": row.report_id,
                    "sentence_number": row.sentence_number,
                    "llama_score": 4
                }
            
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) + (attempt * 0.5)
                print(f"Llama retry {attempt + 1}/{max_retries} after {wait_time}s: {error_str[:50]}")
                sleep(wait_time)
                continue
            else:
                print(f"Llama failed after {max_retries} attempts: {error_str[:100]}")
                # Return default score 4 on final failure (matches working version)
                return {
                    "report_id": row.report_id,
                    "sentence_number": row.sentence_number,
                    "llama_score": 4
                }

print("✅ Llama prompt functions defined")



In [ ]:
# ==================================================
# RUN LLAMA EXPERIMENTS (RUNNING FIRST - TO CATCH ERRORS EARLY)
# ZERO, ONE, TWO, THREE-SHOT
# ==================================================
print("🔬 Running Llama few-shot experiments...")
print("=" * 60)

def process_llama_in_batches(windows_list, prompt_template, workers=5, batch_size=20):
    """Process in batches to avoid overwhelming the endpoint - reduced workers and batch size for stability"""
    all_results = []
    total = len(windows_list)
    
    for batch_start in range(0, total, batch_size):
        batch_end = min(batch_start + batch_size, total)
        batch = windows_list[batch_start:batch_end]
        
        print(f"Llama: Processing batch {batch_start}-{batch_end}/{total}")
        
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = [executor.submit(score_llama_with_retry, row, prompt_template) for row in batch]
            for i, future in enumerate(as_completed(futures)):
                if (batch_start + i) % 10 == 0 or (batch_start + i) == total - 1:
                    print(f"Llama: {batch_start + i + 1}/{total}")
                all_results.append(future.result())
        
        # Longer pause between batches to avoid overwhelming endpoint
        if batch_end < total:
            sleep(3)
    
    return all_results

# Zero-shot
print("\n🔬 Running ZERO-SHOT experiment...")
llama_prompt_zero = create_llama_prompt_zero_shot()
llama_zero_shot_results = process_llama_in_batches(windows_list, llama_prompt_zero, workers=5, batch_size=20)
print(f"✅ Llama Zero-shot complete: {len(llama_zero_shot_results)} results")

llama_zero_shot_df = spark.createDataFrame(llama_zero_shot_results)

# One-shot
print("\n🔬 Running ONE-SHOT experiment...")
llama_prompt_one = create_llama_prompt_one_shot()
llama_one_shot_results = process_llama_in_batches(windows_list, llama_prompt_one, workers=5, batch_size=20)
print(f"✅ Llama One-shot complete: {len(llama_one_shot_results)} results")

llama_one_shot_df = spark.createDataFrame(llama_one_shot_results)

# Two-shot
print("\n🔬 Running TWO-SHOT experiment...")
llama_prompt_two = create_llama_prompt_two_shot()
llama_two_shot_results = process_llama_in_batches(windows_list, llama_prompt_two, workers=5, batch_size=20)
print(f"✅ Llama Two-shot complete: {len(llama_two_shot_results)} results")

llama_two_shot_df = spark.createDataFrame(llama_two_shot_results)

# Three-shot
print("\n🔬 Running THREE-SHOT experiment...")
llama_prompt_three = create_llama_prompt_three_shot()
llama_three_shot_results = process_llama_in_batches(windows_list, llama_prompt_three, workers=5, batch_size=20)
print(f"✅ Llama Three-shot complete: {len(llama_three_shot_results)} results")

llama_three_shot_df = spark.createDataFrame(llama_three_shot_results)

print("\n✅ All Llama experiments complete!")



In [ ]:
# ==================================================
# CLAUDE BATCH PROCESSING FUNCTION
# ==================================================
BATCH_SIZE = 5

def process_claude_batch(batch, prompt_template):
    """Process a batch of windows with Claude"""
    valid_batch = [row for row in batch if row.window_text is not None]
    
    if not valid_batch:
        return [{
            "report_id": row.report_id,
            "sentence_number": row.sentence_number,
            "claude_score": None
        } for row in batch]
    
    items_str = "\n".join([f"[{row.report_id}|{row.sentence_number}]: {row.window_text[:1500]}" for row in valid_batch])
    
    try:
        response = claude_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=500,
            messages=[{"role": "user", "content": prompt_template.format(items=items_str)}]
        )
        
        response_text = response.content[0].text.strip()
        
        if "token" in response_text.lower() and ("limit" in response_text.lower() or "exceeded" in response_text.lower()):
            return [{
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "claude_score": "Error" if row.window_text is not None else None
            } for row in batch]
        
        try:
            results = json.loads(response_text)
        except json.JSONDecodeError:
            json_match = re.search(r'\[.*\]', response_text, re.DOTALL)
            if json_match:
                try:
                    results = json.loads(json_match.group())
                except:
                    return [{
                        "report_id": row.report_id,
                        "sentence_number": row.sentence_number,
                        "claude_score": "Error" if row.window_text is not None else None
                    } for row in batch]
            else:
                return [{
                    "report_id": row.report_id,
                    "sentence_number": row.sentence_number,
                    "claude_score": "Error" if row.window_text is not None else None
                } for row in batch]
        
        if not isinstance(results, list):
            return [{
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "claude_score": "Error" if row.window_text is not None else None
            } for row in batch]
        
        output = []
        for idx, item in enumerate(results):
            try:
                if "id" not in item or "score" not in item:
                    continue
                
                item_id = item["id"].strip("[]")
                parts = item_id.split("|")
                
                if len(parts) == 2:
                    report_id = int(parts[0])
                    sentence_number = int(parts[1])
                elif len(parts) == 1 and idx < len(valid_batch):
                    row = valid_batch[idx]
                    report_id = row.report_id
                    sentence_number = row.sentence_number
                else:
                    continue
                
                score = float(item["score"])
                
                if not (0 <= score <= 7):
                    output.append({
                        "report_id": report_id,
                        "sentence_number": sentence_number,
                        "claude_score": "Error"
                    })
                else:
                    output.append({
                        "report_id": report_id,
                        "sentence_number": sentence_number,
                        "claude_score": score
                    })
            except (ValueError, IndexError):
                continue
        
        score_map = {(item["report_id"], item["sentence_number"]): item["claude_score"] 
                     for item in output}
        
        final_output = []
        for row in batch:
            key = (row.report_id, row.sentence_number)
            if row.window_text is None:
                score = None
            elif key in score_map:
                score = score_map[key]
            else:
                score = "Error"
            
            final_output.append({
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "claude_score": score
            })
        
        return final_output
        
    except Exception as e:
        error_msg = str(e).lower()
        is_token_error = "token" in error_msg and ("limit" in error_msg or "exceeded" in error_msg)
        is_timeout = "timeout" in error_msg or "timed out" in error_msg
        is_api_error = "api" in error_msg or "rate limit" in error_msg or "429" in error_msg
        
        if is_token_error or is_timeout or is_api_error:
            return [{
                "report_id": row.report_id,
                "sentence_number": row.sentence_number,
                "claude_score": "Error" if row.window_text is not None else None
            } for row in batch]
        
        return [{
            "report_id": row.report_id,
            "sentence_number": row.sentence_number,
            "claude_score": "Error" if row.window_text is not None else None
        } for row in batch]

print("✅ Claude batch processing function defined")



In [ ]:
# ==================================================
# RUN CLAUDE EXPERIMENTS (RUNNING SECOND - AFTER LLAMA)
# ZERO, ONE, TWO, THREE-SHOT
# ==================================================
print("🔬 Running Claude few-shot experiments...")
print("=" * 60)

# Define BATCH_SIZE if not already defined (in case cell 11 wasn't run)
if 'BATCH_SIZE' not in globals():
    BATCH_SIZE = 5

claude_batches = [windows_list[i:i + BATCH_SIZE] for i in range(0, len(windows_list), BATCH_SIZE)]

# Zero-shot
print("\n🔬 Running ZERO-SHOT experiment...")
prompt_zero = create_prompt_zero_shot()
zero_shot_results = []

with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(process_claude_batch, batch, prompt_zero): batch for batch in claude_batches}
    for i, future in enumerate(as_completed(futures)):
        if i % 20 == 0:
            print(f"Zero-shot: Processed {i}/{len(claude_batches)} batches")
        zero_shot_results.extend(future.result())

zero_shot_df = spark.createDataFrame(zero_shot_results)
print(f"✅ Zero-shot complete: {len(zero_shot_results)} results")

# One-shot
print("\n🔬 Running ONE-SHOT experiment...")
prompt_one = create_prompt_one_shot()
one_shot_results = []

with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(process_claude_batch, batch, prompt_one): batch for batch in claude_batches}
    for i, future in enumerate(as_completed(futures)):
        if i % 20 == 0:
            print(f"One-shot: Processed {i}/{len(claude_batches)} batches")
        one_shot_results.extend(future.result())

one_shot_df = spark.createDataFrame(one_shot_results)
print(f"✅ One-shot complete: {len(one_shot_results)} results")

# Two-shot
print("\n🔬 Running TWO-SHOT experiment...")
prompt_two = create_prompt_two_shot()
two_shot_results = []

with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(process_claude_batch, batch, prompt_two): batch for batch in claude_batches}
    for i, future in enumerate(as_completed(futures)):
        if i % 20 == 0:
            print(f"Two-shot: Processed {i}/{len(claude_batches)} batches")
        two_shot_results.extend(future.result())

two_shot_df = spark.createDataFrame(two_shot_results)
print(f"✅ Two-shot complete: {len(two_shot_results)} results")

# Three-shot
print("\n🔬 Running THREE-SHOT experiment...")
prompt_three = create_prompt_three_shot()
three_shot_results = []

with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(process_claude_batch, batch, prompt_three): batch for batch in claude_batches}
    for i, future in enumerate(as_completed(futures)):
        if i % 20 == 0:
            print(f"Three-shot: Processed {i}/{len(claude_batches)} batches")
        three_shot_results.extend(future.result())

three_shot_df = spark.createDataFrame(three_shot_results)
print(f"✅ Three-shot complete: {len(three_shot_results)} results")

print("\n✅ All Claude experiments complete!")



In [ ]:
# ==================================================
# COMBINE ALL RESULTS FOR COMPARISON
# ==================================================
print("🔄 Combining all results...")

# Prepare original dataframe
original_df = (df
    .withColumn("report_id", F.col("`doc #`").cast("long"))
    .withColumn("sentence_number", F.col("`sentence #`").cast("long"))
)

# Join all results with original data
comparison_df = (original_df
    # Claude results
    .join(zero_shot_df.withColumnRenamed("claude_score", "claude_zero_shot_score"), ["report_id", "sentence_number"], "left")
    .join(one_shot_df.withColumnRenamed("claude_score", "claude_one_shot_score"), ["report_id", "sentence_number"], "left")
    .join(two_shot_df.withColumnRenamed("claude_score", "claude_two_shot_score"), ["report_id", "sentence_number"], "left")
    .join(three_shot_df.withColumnRenamed("claude_score", "claude_three_shot_score"), ["report_id", "sentence_number"], "left")

    # Llama results
    .join(llama_zero_shot_df.withColumnRenamed("llama_score", "llama_zero_shot_score"), ["report_id", "sentence_number"], "left")
    .join(llama_one_shot_df.withColumnRenamed("llama_score", "llama_one_shot_score"), ["report_id", "sentence_number"], "left")
    .join(llama_two_shot_df.withColumnRenamed("llama_score", "llama_two_shot_score"), ["report_id", "sentence_number"], "left")
    .join(llama_three_shot_df.withColumnRenamed("llama_score", "llama_three_shot_score"), ["report_id", "sentence_number"], "left")
    .select(
        F.col("`global #`").alias("global_num"),
        F.col("`doc #`").alias("doc_num"),
        F.col("`sentence #`").alias("sentence_num"),
        F.col("`firm name`").alias("firm_name"),
        F.col("`analyst`").alias("analyst"),
        F.col("`date`").alias("date"),
        F.col("sentence").alias("sentence_text"),
        
        # Claude scores
        F.col("claude_zero_shot_score"),
        F.col("claude_one_shot_score"),
        F.col("claude_two_shot_score"),
        F.col("claude_three_shot_score"),
        
        # Llama scores
        F.col("llama_zero_shot_score"),
        F.col("llama_one_shot_score"),
        F.col("llama_two_shot_score"),
        F.col("llama_three_shot_score")
    )
    .orderBy("doc_num", "sentence_num")
)

print(f"✅ Comparison dataframe created: {comparison_df.count()} rows")
display(comparison_df.limit(5))



In [ ]:
# ==================================================
# SAVE RESULTS
# ==================================================
table_name = "two_models_few_shot_comparison"
print(f"Saving results to table: {table_name}...")

comparison_df.write.mode("overwrite").saveAsTable(table_name)

print(f"✅ Results saved to table '{table_name}'")
print("You can now query this table in SQL or other notebooks.")
